# CryoFree: choosing the excitation frequency, and normal vs streamed sweeps

dmz_measure is a fork of measureme. `dm.Station` / `dm.AsyncStation` are measureme's
stations (`fp`, `plot`, `add_comment`, `sweep`, `megasweep`, `watch`, `ramp` unchanged),
with streaming added:

1. **Survey** (`s.noise_survey`): noise spectrum of every channel from one streamed record.
2. **Choose** (`dm.best_frequencies`, `dm.expected_noise`): quietest frequencies for the
   lock-in settings you will use, including how steady the noise is over time.
3. **Check** (`s.check_frequencies`): measure chosen frequencies with the excitation on.
4. **Sweeps:** `s.sweep` (normal) vs `s.stream_sweep` (streamed) over the back gate.

The code changes reference frequency and filter settings and restores them afterwards.
It **never changes the excitation amplitude**: switch it off and on by hand where noted.
Run this from the MM03_CF folder.

In [ ]:
import sweep
import numpy as np
import pyvisa
import time
import qcodes as qc
import matplotlib.pyplot as plt
%matplotlib inline

from qcodes.instrument_drivers.stanford_research import SR860
from qcodes.instrument_drivers.stanford_research import SR830
from qcodes.instrument_drivers.Keithley import Keithley2450
from qcodes.instrument_drivers.Keithley import Keithley2400
from qcodes_contrib_drivers.drivers.Cryogenic.CryogenicSMS120C import CryogenicSMS120C

import dmz_measure as dm

In [ ]:
sweep.set_basedir('./data_03+12')

In [ ]:
SR860_A = SR860('SR860_A', 'TCPIP::192.168.50.11::inst0::INSTR')
SR860_B = SR860('SR860_B', 'TCPIP::192.168.50.12::inst0::INSTR')
SR860_C = SR860('SR860_C', 'TCPIP::192.168.50.13::inst0::INSTR')
SR860_D = SR860('SR860_D', 'TCPIP::192.168.50.14::inst0::INSTR')
SR860_E = SR860('SR860_E', 'TCPIP::192.168.50.15::inst0::INSTR')
SR860_F = SR860('SR860_F', 'TCPIP::192.168.50.16::inst0::INSTR')
SR830_G = SR830('SR830_G', 'GPIB0::17::INSTR')
SR830_H = SR830('SR830_H', 'GPIB0::18::INSTR')
SR830_I = SR830('SR830_I', 'GPIB0::19::INSTR')
SR830_T = SR830('SR830_T', 'GPIB0::20::INSTR')

In [ ]:
Kbg1 = Keithley2400('Kbg1', 'GPIB0::23::INSTR', timeout=30)
Ktg1 = Keithley2400('Ktg1', 'GPIB0::24::INSTR', timeout=30)
Kbg2 = Keithley2450('Kbg2', 'TCPIP::192.168.50.21::inst0::INSTR', timeout=30)
Ktg2 = Keithley2450('Ktg2', 'TCPIP::192.168.50.22::inst0::INSTR', timeout=30)

In [ ]:
Kbg1.output(0)
Kbg1.mode('VOLT')
Kbg1.sense('CURR')
Kbg1.rangev(20)
Kbg1.rangei(1E-6)
Kbg1.compliancei(1E-6)
Kbg1.volt.step = 0.005
Kbg1.volt.inter_delay = 0.005

Kbg1.output(1)

Ktg1.output(0)
Ktg1.mode('VOLT')
Ktg1.sense('CURR')
Ktg1.rangev(20)
Ktg1.rangei(1E-6)
Ktg1.compliancei(1E-6)
Ktg1.volt.step = 0.005
Ktg1.volt.inter_delay = 0.005

Ktg1.output(1)

Kbg2.source.function('voltage')
Kbg2.output_enabled(1)
Kbg2.source.range(21)
Kbg2.sense.range(1E-6)
Kbg2.source.voltage.step = 0.005
Kbg2.source.voltage.inter_delay = 0.005
Kbg2.source.block_during_ramp(True)

Kbg2.volt = Kbg2.source.voltage
Kbg2.curr = Kbg2.sense.current

Ktg2.source.function('voltage')
Ktg2.output_enabled(1)
Ktg2.source.range(21)
Ktg2.sense.range(1E-6)
Ktg2.source.voltage.step = 0.005
Ktg2.source.voltage.inter_delay = 0.005
Ktg2.source.block_during_ramp(True)

Ktg2.volt = Ktg2.source.voltage
Ktg2.curr = Ktg2.sense.current

In [ ]:
magnet = CryogenicSMS120C(
    name='magnet',
    address='ASRL23::INSTR',
    coil_constant=0.0983606557,  # T/A
    current_rating=91.5,    # Amps
    current_ramp_limit=0.002/0.0983606557 # A/s
)

## Measurement config

In [ ]:
measurement_config = {
    'SR860_A': '9-12',
    'SR860_B': '15-14',
    'SR860_C': '15-11',
    'Vbg2': '13',
    'Vtg2': '16'
}

SR860s = [(SR860_A, 1E6), (SR860_B, 100), (SR860_C, 100)]

## 1. Noise survey
**Turn the excitation off by hand first** (the code never changes it). Every SR860 in
`measurement_config` streams with the reference at `f_ref`, τ = 100 µs and sync off. The
complex FFT of X + iY gives each channel's input noise spectrum; the gains come from `s.fp`.

In [ ]:
s = dm.AsyncStation(measurement_config)
s.add_comment('noise survey, excitation off')
for SR860, gain in SR860s:
    s.fp(SR860.X, gain=gain)
    s.fp(SR860.Y, gain=gain)

survey = s.noise_survey(
    current='SR860_A',  #     lock-in that measures the current
    f_ref=100,          # Hz  reference frequency during the survey
    duration=600,       # s   streaming time (longer = smoother spectra, more time blocks)
    fmin=0, fmax=200,   # Hz  band to keep (anything from 0 to f_ref + rate/2 works)
    resolution=0.05,   # Hz  FFT bin width: Welch segments of 1/resolution s, 50 % overlap
    tau=100e-6,         # s   lock-in time constant during the survey (<= 100 us keeps the band flat)
    rate=10e3,          # Hz  stream rate
)
survey.plot();

In [ ]:
survey.spectrogram('SR860_B', block=30);   # noise vs time: look for bursts (compressor, pumps, ...)

## 2. Choose frequencies (analysis only; rerun as often as you like)

`best_frequencies` predicts the noise of one data point at every reference frequency in
`[fmin, fmax]`, for the filter that makes one data point:
* **Normal measurement (`s.sweep`):** the SR860 time constant (`tau`, `slope`, `advanced`
  = the Advanced/Gaussian filter, `sync`), read once (`dwell=None`).
* **Streamed (`s.stream_sweep`):** a short `tau` with `sync=True`, plus a software average
  over `dwell` seconds, weighted with `window` (`'hann'` or `'boxcar'`).

To account for noise that comes and goes, the survey is cut into `block`-second pieces and
the prediction is made for each piece. `stat='p90'` ranks by the noise exceeded 10 % of
the time. In the table, **spread** = worst block / typical block; the line above the table
gives the value steady noise produces. `expected_noise` gives the numbers for frequencies
you type in. A saved survey can be reloaded with `survey = dm.load_survey(<run ID>)`.

In [ ]:
best = dm.best_frequencies(
    survey,
    fmin=10, fmax=40,       # Hz  range to search
    tau=0.3, slope=24,      # lock-in time constant (s) and slope (dB/oct) you will measure with
    advanced=False,         #     SR860 Advanced (Gaussian) filter on?  check: SR860_A.ask('ADVFILT?')
    sync=False,             #     sync filter
    dwell=None,             # s   software average per data point (None = one reading)
    window='boxcar',        #     weighting of that average: 'boxcar' or 'hann'
    n=5, min_spacing=1.0,   #     how many to return, minimum spacing (Hz)
    avoid_mains=True,       #     skip mains * p/q (15, 20, 30, 40, 45 ... Hz at 60 Hz mains)
    block=60, stat='p90',  # s   time blocks; rank by 'mean', 'median', 'p90' or 'max'
)
best.plot();

In [ ]:
freqs = [SR860_A.frequency(), 17.777, 23.3]   # Hz, any frequencies

# normal measurement: one reading with the SR860 filter
dm.expected_noise(survey, freqs, tau=0.3, slope=24, advanced=False, sync=False, dwell=None,
                  block=60)
# streamed: tau ~ 1 ms + sync, Hann-weighted 0.3 s average per point
dm.expected_noise(survey, freqs, tau=1e-3, slope=24, sync=True, dwell=0.3, window='hann',
                  block=60);

## 3. Check frequencies (excitation on)
**Turn the excitation back on by hand** and set a representative gate point. Reported per
channel: R = V/I (I for the current channel), the noise of one data point (scatter of
`dwell` averages, relative), the worst `block` stretch, and the phase.

In [ ]:
s.ramp(Ktg2.volt, 1)
s.ramp(Kbg2.volt, 1)

In [ ]:
check = s.check_frequencies(
    best.frequencies[:3] + [SR860_A.frequency()],   # any list (Hz): here the top 3 + the one in use
    current='SR860_A',
    tau=None,           # s   None = largest SR860 value <= 1/(30 f) for each frequency
    slope=24,           #     dB/oct
    advanced=None,      #     Advanced filter on/off; None = leave as set
    sync=True,          #     sync filter
    duration=120,       # s   streamed per frequency
    dwell=0.3,          # s   averaging per data point; noise = scatter of these averages
    window='hann',      #     weighting of that average: 'hann' or 'boxcar'
    block=30,           # s   stretch length for the 'worst' column
)
check.plot();   # Allan deviation: keeps falling as 1/sqrt(time) while averaging still helps

In [ ]:
s.ramp(Ktg2.volt, 0)
s.ramp(Kbg2.volt, 0)

## 4. Back-gate line sweeps: normal vs streamed

The same station does both. `s.sweep` is measureme's sweep, unchanged: set the gate, wait
`delay`, read every followed parameter once, with the lock-ins exactly as they are set.

`s.stream_sweep` writes the same columns, live plots and files. The followed SR860
X/Y/R/P values come from the stream: τ ≈ 1/(30 f) + sync, and each point is the
`window`-weighted average of the `dwell` seconds after `settle`. Every other followed
parameter (gate leakage etc.) is read as usual. The lock-in settings are restored
afterwards and the excitation is never touched. The stream is saved with the run, so
`dm.refilter(id, ...)` can redo the averaging later.

Each method sweeps up and back down. Lag shows up as a mismatch between the two directions
(`fwd/rev` in the `dm.compare` table).

Set the frequency you chose and the time constant you normally use by hand first.

In [ ]:
s = dm.AsyncStation(measurement_config)
s.add_comment('back-gate line sweeps: normal vs streamed')
for SR860, gain in SR860s:
    s.fp(SR860.X, gain=gain)
    s.fp(SR860.Y, gain=gain)
    s.plot(Kbg2.volt, SR860.X)
s.fp(Kbg2.curr)
s.fp(Ktg2.curr)

Vbg = np.linspace(-1, 1, 201)   # V

### Where the time goes, and ramping faster without bigger steps
qcodes cuts each gate set into chunks of the Keithley's `step` (5 mV), at least
`inter_delay` (5 ms) apart. With `block_during_ramp(True)`, every chunk also waits for a
`*OPC?` reply from the 2450, so each 5 mV chunk costs a full round trip on top of the 5 ms.
`s.timing` measures this (set, `*OPC?`, each read) and gives the real total per sweep.

**`dm.fast_ramp(Kbg2.volt)`** keeps the same 5 mV chunks, at least 5 ms apart, but sends them
without waiting for a reply each time, and asks `*OPC?` once when the whole set is sent: the
call still returns only when the gate is at the new value, so the ramp is never faster than
before. (`dm.gate_steps(..., max_step=...)` would instead make each point one bigger step.)

`s.step_response` measures how long after a gate step the signal has settled (gate line +
device + lock-in), and `dm.step_response(run_id)` measures the same from a streamed sweep's
own steps. Use it as `settle`. In streamed sweeps the other followed parameters (leakage
currents) are read during the dwell, so they cost no extra time.

In [ ]:
s.timing(Kbg2.volt, Vbg, n=20, delay=0.6, dwell=0.3)          # as now: *OPC? after every 5 mV chunk
with dm.fast_ramp(Kbg2.volt):                                  # same chunks, one *OPC? per set
    s.timing(Kbg2.volt, Vbg, n=20, delay=0.6, dwell=0.3)

In [ ]:
# the gate moves as in the sweep (5 mV chunks, fast_ramp): pick a step like a sweep point
# (one or a few increments) that still changes R clearly
with dm.fast_ramp(Kbg2.volt):
    st = s.step_response(Kbg2.volt, 0.0, 0.05)    # V
st.plot();

In [ ]:
# normal (measureme): lock-ins as set, wait `delay` after each step, read once
with dm.fast_ramp(Kbg2.volt):     # same 5 mV chunks, one *OPC? per point
    old_up   = s.sweep(Kbg2.volt, Vbg, delay=0.6)
    old_down = s.sweep(Kbg2.volt, Vbg[::-1], delay=0.6)

In [ ]:
# streamed: same station, same followed parameters
kw = dict(
    dwell=0.3,          # s   averaged per point
    settle=st.settle,   # s   skipped after each step (measured above; None = lock-in settling + 1 period)
    window='hann',      #     'hann' or 'boxcar'
    tau=None,           # s   None = largest SR860 value <= 1/(30 f)
    slope=24, advanced=None, sync=True,
)
with dm.fast_ramp(Kbg2.volt):
    new_up   = s.stream_sweep(Kbg2.volt, Vbg, **kw)
    new_down = s.stream_sweep(Kbg2.volt, Vbg[::-1], **kw)

In [ ]:
# the same measured from the streamed sweep's own steps; if it needs more settle than was used,
# dm.refilter(new_up.id, settle=...) re-averages the saved stream
dm.step_response(new_up.id).plot();

In [ ]:
dm.compare([old_up.id, old_down.id, new_up.id, new_down.id], current='SR860_A');

In [ ]:
# the raw stream is saved with the run: re-average the same data, no re-measuring
d = dm.refilter(new_up.id, dwell=0.1, window='boxcar')      # same format as sweep_load.pload1d
R = lambda d, n: np.real((d[n + '_X'] + 1j * d[n + '_Y']) / (d['SR860_A_X'] + 1j * d['SR860_A_Y']))
d0 = dm.refilter(new_up.id)                                  # the original averaging
plt.plot(d0['xs'], R(d0, 'SR860_B'), '.-', label='dwell 0.3 s, Hann')
plt.plot(d['xs'], R(d, 'SR860_B'), '.-', label='dwell 0.1 s, plain mean')
plt.xlabel('Vbg (V)'); plt.ylabel('R (Ohm)'); plt.legend();

In [ ]:
s.ramp(Kbg2.volt, 0)

## 5. Streamed gate map

`s.stream_megasweep` is `s.megasweep` with each fast line streamed:
```python
s.stream_megasweep(Kbg2.volt, np.linspace(-2.25, 5.5, 201), Ktg2.volt, np.linspace(-6, 6, 201),
                   slow_delay=3, dwell=0.3, settle=None, window='hann')
```